Phase 1  Project Setup              ██████████  DONE
Phase 2  Dataset Understanding      ██████░░░░  IN PROGRESS
Phase 3  Advanced EDA               ░░░░░░░░░░
Phase 4  Data Cleaning              ░░░░░░░░░░
Phase 5  Feature Engineering        ░░░░░░░░░░
Phase 6  Preprocessing              ░░░░░░░░░░
Phase 7  Baseline K-Means           ░░░░░░░░░░
Phase 8  PyTorch Autoencoder        ░░░░░░░░░░
Phase 9  Latent Embeddings          ░░░░░░░░░░
Phase 10 Latent K-Means             ░░░░░░░░░░
Phase 11 Customer Profiling         ░░░░░░░░░░
Phase 12 Visualization              ░░░░░░░░░░
Phase 13 Explainability             ░░░░░░░░░░
Phase 14 Inference Pipeline         ░░░░░░░░░░
Phase 15 Streamlit                  ░░░░░░░░░░
Phase 16 Production Packaging       ░░░░░░░░░░
Phase 17 GitHub Documentation       ░░░░░░░░░░
Phase 18 Interview Preparation      ░░░░░░░░░░

# Phase 1 Project Setup

# Phase 2 — Dataset Understanding & Relational Data Analysis

In [15]:
import sys

print(sys.version)
print(sys.executable)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\venv\Scripts\python.exe


In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.5.1
Pandas version: 3.0.5


In [17]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Raw data exists:", RAW_DATA_DIR.exists())

Project root: C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation
Raw data directory: C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\data\raw
Raw data exists: True


In [18]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [19]:
DATA_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

datasets = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in DATA_FILES.items()
}

print(f"Successfully loaded {len(datasets)} datasets.")

Successfully loaded 9 datasets.


In [20]:
for name, df in datasets.items():
    print(
        f"{name:<22} "
        f"Rows: {df.shape[0]:>10,} | "
        f"Columns: {df.shape[1]:>2}"
    )

customers              Rows:     99,441 | Columns:  5
geolocation            Rows:  1,000,163 | Columns:  5
orders                 Rows:     99,441 | Columns:  8
order_items            Rows:    112,650 | Columns:  7
payments               Rows:    103,886 | Columns:  5
reviews                Rows:     99,224 | Columns:  7
products               Rows:     32,951 | Columns:  9
sellers                Rows:      3,095 | Columns:  4
category_translation   Rows:         71 | Columns:  2


In [21]:
summary_records = []

for name, df in datasets.items():
    summary_records.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )
    })

dataset_summary = (
    pd.DataFrame(summary_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

dataset_summary

,dataset,rows,columns,duplicate_rows,missing_cells,memory_mb
0,geolocation,1000163,5,261831,0,50.12
1,order_items,112650,7,0,0,18.37
2,payments,103886,5,0,0,8.11
3,customers,99441,5,0,0,11.03
4,orders,99441,8,0,4908,21.95
5,reviews,99224,7,0,145903,17.84
6,products,32951,9,0,2448,3.73
7,sellers,3095,4,0,0,0.22
8,category_translation,71,2,0,0,0.00


In [22]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    schema = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "unique_values": df.nunique(dropna=False).values
    })

    display(schema)


CUSTOMERS


,column,dtype,non_null,null_count,unique_values
0,customer_id,str,99441,0,99441
1,customer_unique_id,str,99441,0,96096
2,customer_zip_code_prefix,int64,99441,0,14994
3,customer_city,str,99441,0,4119
4,customer_state,str,99441,0,27



GEOLOCATION


,column,dtype,non_null,null_count,unique_values
0,geolocation_zip_code_prefix,int64,1000163,0,19015
1,geolocation_lat,float64,1000163,0,717360
2,geolocation_lng,float64,1000163,0,717613
3,geolocation_city,str,1000163,0,8011
4,geolocation_state,str,1000163,0,27



ORDERS


...

,column,dtype,non_null,null_count,unique_values
0,product_id,str,32951,0,32951
1,product_category_name,str,32341,610,74
2,product_name_lenght,float64,32341,610,67
3,product_description_lenght,float64,32341,610,2961
4,product_photos_qty,float64,32341,610,20
5,product_weight_g,float64,32949,2,2205
6,product_length_cm,float64,32949,2,100
7,product_height_cm,float64,32949,2,103
8,product_width_cm,float64,32949,2,96



SELLERS


,column,dtype,non_null,null_count,unique_values
0,seller_id,str,3095,0,3095
1,seller_zip_code_prefix,int64,3095,0,2246
2,seller_city,str,3095,0,611
3,seller_state,str,3095,0,23



CATEGORY_TRANSLATION


,column,dtype,non_null,null_count,unique_values
0,product_category_name,str,71,0,71
1,product_category_name_english,str,71,0,71


In [23]:
key_checks = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id", "customer_id"],
    "order_items": ["order_id", "product_id", "seller_id"],
    "payments": ["order_id"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

for dataset_name, columns in key_checks.items():
    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    for column in columns:
        print(
            f"{column:<25} "
            f"Rows: {len(df):>8,} | "
            f"Unique: {df[column].nunique():>8,} | "
            f"Missing: {df[column].isna().sum():>6,}"
        )


CUSTOMERS
customer_id               Rows:   99,441 | Unique:   99,441 | Missing:      0
customer_unique_id        Rows:   99,441 | Unique:   96,096 | Missing:      0

ORDERS
order_id                  Rows:   99,441 | Unique:   99,441 | Missing:      0
customer_id               Rows:   99,441 | Unique:   99,441 | Missing:      0

ORDER_ITEMS
order_id                  Rows:  112,650 | Unique:   98,666 | Missing:      0
product_id                Rows:  112,650 | Unique:   32,951 | Missing:      0
seller_id                 Rows:  112,650 | Unique:    3,095 | Missing:      0

PAYMENTS
order_id                  Rows:  103,886 | Unique:   99,440 | Missing:      0

REVIEWS
review_id                 Rows:   99,224 | Unique:   98,410 | Missing:      0
order_id                  Rows:   99,224 | Unique:   98,673 | Missing:      0

PRODUCTS
product_id                Rows:   32,951 | Unique:   32,951 | Missing:      0

SELLERS
seller_id                 Rows:    3,095 | Unique:    3,095 | Missing:  

In [24]:
customers = datasets["customers"]

customer_identity = pd.Series({
    "rows": len(customers),
    "unique_customer_ids": customers["customer_id"].nunique(),
    "unique_customers": customers["customer_unique_id"].nunique(),
    "duplicate_customer_ids": customers["customer_id"].duplicated().sum(),
    "duplicate_unique_customer_ids": customers["customer_unique_id"].duplicated().sum()
})

customer_identity

rows                             99441
unique_customer_ids              99441
unique_customers                 96096
duplicate_customer_ids               0
duplicate_unique_customer_ids     3345
dtype: int64

In [25]:
orders_per_unique_customer = (
    customers
    .groupby("customer_unique_id")["customer_id"]
    .nunique()
    .sort_values(ascending=False)
)

orders_per_unique_customer.describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_id, dtype: float64

In [26]:
orders_per_unique_customer.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
Name: customer_id, dtype: int64

In [27]:
orders = datasets["orders"]

orders_customer_check = pd.Series({
    "orders": len(orders),
    "unique_order_ids": orders["order_id"].nunique(),
    "unique_customer_ids_in_orders": orders["customer_id"].nunique(),
    "customer_ids_missing": orders["customer_id"].isna().sum(),
    "orders_with_unknown_customer": (
        ~orders["customer_id"].isin(customers["customer_id"])
    ).sum()
})

orders_customer_check

orders                           99441
unique_order_ids                 99441
unique_customer_ids_in_orders    99441
customer_ids_missing                 0
orders_with_unknown_customer         0
dtype: int64

In [29]:
order_relationship_checks = []

for table_name in ["order_items", "payments", "reviews"]:
    df = datasets[table_name]

    order_relationship_checks.append({
        "table": table_name,
        "rows": len(df),
        "unique_order_ids": df["order_id"].nunique(),
        "unknown_order_ids": (
            ~df["order_id"].isin(orders["order_id"])
        ).sum()
    })

pd.DataFrame(order_relationship_checks)

,table,rows,unique_order_ids,unknown_order_ids
0,order_items,112650,98666,0
1,payments,103886,99440,0
2,reviews,99224,98673,0


In [30]:
items_per_order = (
    datasets["order_items"]
    .groupby("order_id")
    .size()
)

payments_per_order = (
    datasets["payments"]
    .groupby("order_id")
    .size()
)

reviews_per_order = (
    datasets["reviews"]
    .groupby("order_id")
    .size()
)

cardinality_summary = pd.DataFrame({
    "order_items": items_per_order.describe(),
    "payments": payments_per_order.describe(),
    "reviews": reviews_per_order.describe()
})

cardinality_summary

,order_items,payments,reviews
count,98666.000000,99440.000000,98673.000000
mean,1.141731,1.044710,1.005584
std,0.538452,0.381166,0.075060
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,21.000000,29.000000,3.000000


In [31]:
order_items = datasets["order_items"]
products = datasets["products"]
sellers = datasets["sellers"]

pd.Series({
    "unknown_product_ids": (
        ~order_items["product_id"].isin(products["product_id"])
    ).sum(),

    "unknown_seller_ids": (
        ~order_items["seller_id"].isin(sellers["seller_id"])
    ).sum()
})

unknown_product_ids    0
unknown_seller_ids     0
dtype: int64

In [32]:
items_per_order = (
    datasets["order_items"]
    .groupby("order_id")
    .size()
)

payments_per_order = (
    datasets["payments"]
    .groupby("order_id")
    .size()
)

reviews_per_order = (
    datasets["reviews"]
    .groupby("order_id")
    .size()
)

cardinality_summary = pd.DataFrame({
    "order_items": items_per_order.describe(),
    "payments": payments_per_order.describe(),
    "reviews": reviews_per_order.describe()
})

cardinality_summary

,order_items,payments,reviews
count,98666.000000,99440.000000,98673.000000
mean,1.141731,1.044710,1.005584
std,0.538452,0.381166,0.075060
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,21.000000,29.000000,3.000000


In [33]:
missing_summary = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_summary.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": int(missing_count),
                "missing_pct": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_df = (
    pd.DataFrame(missing_summary)
    .sort_values(
        ["missing_pct", "missing_count"],
        ascending=False
    )
    .reset_index(drop=True)
)

missing_df

,dataset,column,missing_count,missing_pct
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [34]:
missing_by_dataset = []

for name, df in datasets.items():
    missing_by_dataset.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "columns_with_missing": int(df.isna().any().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
        "missing_pct_all_cells": round(
            df.isna().sum().sum() / df.size * 100,
            2
        )
    })

missing_dataset_summary = pd.DataFrame(missing_by_dataset)

missing_dataset_summary

,dataset,rows,columns,columns_with_missing,total_missing_cells,missing_pct_all_cells
0,customers,99441,5,0,0,0.00
1,geolocation,1000163,5,0,0,0.00
2,orders,99441,8,3,4908,0.62
3,order_items,112650,7,0,0,0.00
4,payments,103886,5,0,0,0.00
5,reviews,99224,7,2,145903,21.01
6,products,32951,9,8,2448,0.83
7,sellers,3095,4,0,0,0.00
8,category_translation,71,2,0,0,0.00


In [ ]:
missing_df

In [35]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pct": round(
            df.duplicated().sum() / len(df) * 100, 4
        )
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,dataset,rows,exact_duplicate_rows,duplicate_pct
0,customers,99441,0,0.0000
1,geolocation,1000163,261831,26.1788
2,orders,99441,0,0.0000
3,order_items,112650,0,0.0000
4,payments,103886,0,0.0000
5,reviews,99224,0,0.0000
6,products,32951,0,0.0000
7,sellers,3095,0,0.0000
8,category_translation,71,0,0.0000


In [36]:
primary_key_checks = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

key_duplicate_results = []

for dataset_name, key_columns in primary_key_checks.items():
    df = datasets[dataset_name]

    key_duplicate_results.append({
        "dataset": dataset_name,
        "key": ", ".join(key_columns),
        "rows": len(df),
        "unique_keys": df[key_columns].drop_duplicates().shape[0],
        "duplicate_keys": int(df.duplicated(subset=key_columns).sum())
    })

pd.DataFrame(key_duplicate_results)

,dataset,key,rows,unique_keys,duplicate_keys
0,customers,customer_id,99441,99441,0
1,orders,order_id,99441,99441,0
2,products,product_id,32951,32951,0
3,sellers,seller_id,3095,3095,0


In [37]:
primary_key_checks = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

key_duplicate_results = []

for dataset_name, key_columns in primary_key_checks.items():
    df = datasets[dataset_name]

    key_duplicate_results.append({
        "dataset": dataset_name,
        "key": ", ".join(key_columns),
        "rows": len(df),
        "unique_keys": df[key_columns].drop_duplicates().shape[0],
        "duplicate_keys": int(
            df.duplicated(subset=key_columns).sum()
        )
    })

key_duplicate_df = pd.DataFrame(key_duplicate_results)

key_duplicate_df

,dataset,key,rows,unique_keys,duplicate_keys
0,customers,customer_id,99441,99441,0
1,orders,order_id,99441,99441,0
2,products,product_id,32951,32951,0
3,sellers,seller_id,3095,3095,0


### Primary Key Validation

- `customer_id` uniquely identifies each record in the customers table.
- `order_id` uniquely identifies each order.
- `product_id` uniquely identifies each product.
- `seller_id` uniquely identifies each seller.
- No duplicate values were found in these expected primary keys.
- `customer_unique_id` is intentionally not treated as the primary key of the
  customers table because the same real-world customer may be associated with
  multiple `customer_id` records across different orders.

## 9. Order Status Analysis

Analyze the distribution of order statuses and their relationship with missing
order lifecycle timestamps. This will help determine which transactions should
be considered valid for customer behavioral segmentation.

In [38]:
order_status_summary = (
    datasets["orders"]["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

order_status_summary["percentage"] = (
    order_status_summary["order_count"]
    / len(datasets["orders"])
    * 100
).round(2)

order_status_summary

,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


Missing delivery date by status

In [39]:
orders = datasets["orders"]

delivery_missing_by_status = (
    orders
    .groupby("order_status", dropna=False)
    .agg(
        total_orders=("order_id", "size"),
        missing_delivery_date=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

delivery_missing_by_status["missing_pct"] = (
    delivery_missing_by_status["missing_delivery_date"]
    / delivery_missing_by_status["total_orders"]
    * 100
).round(2)

delivery_missing_by_status.sort_values(
    "total_orders",
    ascending=False
)

,order_status,total_orders,missing_delivery_date,missing_pct
3,delivered,96478,8,0.01
6,shipped,1107,1107,100.00
1,canceled,625,619,99.04
7,unavailable,609,609,100.00
4,invoiced,314,314,100.00
5,processing,301,301,100.00
2,created,5,5,100.00
0,approved,2,2,100.00


In [40]:
delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

print("Delivered orders with missing delivery date:", len(delivered_missing))

delivered_missing.head()

Delivered orders with missing delivery date: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


In [41]:
delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

print(
    "Delivered orders with missing delivery date:",
    len(delivered_missing)
)

delivered_missing

Delivered orders with missing delivery date: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


Inspect current timestamp types

In [42]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [43]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

orders[timestamp_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Find the observation period

In [44]:
observation_window = pd.Series({
    "first_purchase": orders["order_purchase_timestamp"].min(),
    "last_purchase": orders["order_purchase_timestamp"].max(),
    "total_orders": len(orders),
    "unique_purchase_days": orders["order_purchase_timestamp"].dt.date.nunique()
})

observation_window

first_purchase          2016-09-04 21:15:19
last_purchase           2018-10-17 17:30:18
total_orders                          99441
unique_purchase_days                    634
dtype: object

Purchase activity by year

In [45]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

yearly_orders = (
    orders["purchase_year"]
    .value_counts()
    .sort_index()
    .rename_axis("year")
    .reset_index(name="order_count")
)

yearly_orders

,year,order_count
0,2016,329
1,2017,45101
2,2018,54011


Purchase activity by month

In [46]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_orders = (
    orders
    .groupby("purchase_month")
    .size()
    .reset_index(name="order_count")
)

monthly_orders

,purchase_month,order_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


Monthly order distribution

In [47]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_orders = (
    orders
    .groupby("purchase_month")
    .size()
    .reset_index(name="order_count")
)

monthly_orders

,purchase_month,order_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [48]:
# inspect status by month

monthly_status = (
    orders
    .groupby(["purchase_month", "order_status"])
    .size()
    .unstack(fill_value=0)
)

monthly_status

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
purchase_month,,,,,,,,
2016-09,0,2,0,1,0,0,1,0
2016-10,0,24,0,265,18,2,8,7
2016-12,0,0,0,1,0,0,0,0
2017-01,0,3,0,750,12,9,16,10
2017-02,1,17,0,1653,11,32,21,45
2017-03,0,33,0,2546,3,23,45,32
2017-04,1,18,0,2303,14,10,49,9
2017-05,0,29,0,3546,16,23,55,31
2017-06,0,16,0,3135,11,12,47,24


In [49]:
# Check status by month

monthly_status = (
    orders
    .groupby(["purchase_month", "order_status"])
    .size()
    .unstack(fill_value=0)
)

monthly_status.tail(8)

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
purchase_month,,,,,,,,
2018-03,0,26,0,7003,23,9,133,17
2018-04,0,15,0,6798,14,8,99,5
2018-05,0,24,0,6749,24,6,54,16
2018-06,0,18,0,6099,3,0,43,4
2018-07,0,41,0,6159,13,1,60,18
2018-08,0,84,0,6351,23,0,47,7
2018-09,0,15,0,0,0,0,1,0
2018-10,0,4,0,0,0,0,0,0


In [50]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print(f"All orders:       {len(orders):,}")
print(f"Delivered orders: {len(delivered_orders):,}")
print(f"Excluded orders:  {len(orders) - len(delivered_orders):,}")
print(
    f"Retention rate:   "
    f"{len(delivered_orders) / len(orders) * 100:.2f}%"
)

All orders:       99,441
Delivered orders: 96,478
Excluded orders:  2,963
Retention rate:   97.02%


In [51]:
delivered_customer_base = (
    delivered_orders[["order_id", "customer_id"]]
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

customer_population = pd.Series({
    "delivered_orders": len(delivered_customer_base),
    "unique_customer_ids": delivered_customer_base["customer_id"].nunique(),
    "unique_customers": delivered_customer_base["customer_unique_id"].nunique(),
    "missing_customer_unique_id": delivered_customer_base["customer_unique_id"].isna().sum()
})

customer_population

delivered_orders              96478
unique_customer_ids           96478
unique_customers              93358
missing_customer_unique_id        0
dtype: int64

In [52]:
# Examine purchase dates after filtering
pd.Series({
    "first_delivered_purchase":
        delivered_orders["order_purchase_timestamp"].min(),

    "last_delivered_purchase":
        delivered_orders["order_purchase_timestamp"].max()
})

first_delivered_purchase   2016-09-15 12:16:38
last_delivered_purchase    2018-08-29 15:00:37
dtype: datetime64[us]

In [53]:
# Check delivered-order boundaries

delivered_purchase_window = pd.Series({
    "first_delivered_purchase":
        delivered_orders["order_purchase_timestamp"].min(),

    "last_delivered_purchase":
        delivered_orders["order_purchase_timestamp"].max(),

    "unique_purchase_days":
        delivered_orders["order_purchase_timestamp"].dt.date.nunique()
})

delivered_purchase_window

first_delivered_purchase    2016-09-15 12:16:38
last_delivered_purchase     2018-08-29 15:00:37
unique_purchase_days                        612
dtype: object

In [54]:
delivered_orders[
    delivered_orders["order_purchase_timestamp"] >= "2018-09-01"
][
    ["order_id", "order_purchase_timestamp", "order_status"]
].sort_values("order_purchase_timestamp")

,order_id,order_purchase_timestamp,order_status


In [55]:
last_delivered_purchase = delivered_orders[
    "order_purchase_timestamp"
].max()

reference_date = (
    last_delivered_purchase.normalize()
    + pd.Timedelta(days=1)
)

print("Last delivered purchase:", last_delivered_purchase)
print("Reference date:", reference_date)

Last delivered purchase: 2018-08-29 15:00:37
Reference date: 2018-08-30 00:00:00


In [56]:
# Validate Recency boundaries

customer_last_purchase = (
    delivered_customer_base
    .merge(
        delivered_orders[
            ["order_id", "order_purchase_timestamp"]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .groupby("customer_unique_id")[
        "order_purchase_timestamp"
    ]
    .max()
)

recency_days = (
    reference_date - customer_last_purchase
).dt.days

recency_days.describe()

count    93358.000000
mean       237.478888
std        152.595050
min          0.000000
25%        114.000000
50%        218.000000
75%        346.000000
max        713.000000
Name: order_purchase_timestamp, dtype: float64

In [58]:
order_items = datasets["order_items"].copy()

delivered_order_items = order_items[
    order_items["order_id"].isin(delivered_orders["order_id"])
].copy()

print(f"All item records:       {len(order_items):,}")
print(f"Delivered item records: {len(delivered_order_items):,}")
print(
    "Orders represented:    "
    f"{delivered_order_items['order_id'].nunique():,}"
)

All item records:       112,650
Delivered item records: 110,197
Orders represented:    96,478


In [59]:
# Understand item-level monetary fields

delivered_order_items[
    ["price", "freight_value"]
].describe()

,price,freight_value
count,110197.000000,110197.000000
mean,119.980563,19.948598
std,182.299446,15.698136
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.900000,16.260000
75%,134.170000,21.150000
max,6735.000000,409.680000


In [60]:
pd.Series({
    "negative_price":
        (delivered_order_items["price"] < 0).sum(),

    "zero_price":
        (delivered_order_items["price"] == 0).sum(),

    "negative_freight":
        (delivered_order_items["freight_value"] < 0).sum(),

    "zero_freight":
        (delivered_order_items["freight_value"] == 0).sum(),

    "missing_price":
        delivered_order_items["price"].isna().sum(),

    "missing_freight":
        delivered_order_items["freight_value"].isna().sum()
})

negative_price        0
zero_price            0
negative_freight      0
zero_freight        381
missing_price         0
missing_freight       0
dtype: int64

In [61]:
# Build order-item aggregation

order_item_features = (
    delivered_order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        merchandise_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean")
    )
    .reset_index()
)

order_item_features.head()

,order_id,item_count,product_count,seller_count,merchandise_value,freight_value,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90


These features mean:

item_count → number of items in the order
product_count → number of distinct products
seller_count → number of distinct sellers
merchandise_value → total product price
freight_value → total shipping cost
avg_item_price → average item price

In [62]:
order_item_features["order_value"] = (
    order_item_features["merchandise_value"]
    + order_item_features["freight_value"]
)

In [63]:
# Validate the aggregation

print("Rows:", len(order_item_features))
print(
    "Unique orders:",
    order_item_features["order_id"].nunique()
)

print(
    "Duplicate order IDs:",
    order_item_features["order_id"].duplicated().sum()
)

Rows: 96478
Unique orders: 96478
Duplicate order IDs: 0


In [64]:
# verify that aggregation didn't change monetary totals:

validation = pd.Series({
    "raw_merchandise_total":
        delivered_order_items["price"].sum(),

    "aggregated_merchandise_total":
        order_item_features["merchandise_value"].sum(),

    "raw_freight_total":
        delivered_order_items["freight_value"].sum(),

    "aggregated_freight_total":
        order_item_features["freight_value"].sum()
})

validation

raw_merchandise_total           13221498.11
aggregated_merchandise_total    13221498.11
raw_freight_total                2198275.64
aggregated_freight_total         2198275.64
dtype: float64

In [66]:
# Check coverage



pd.Series({
    "delivered_orders": delivered_orders["order_id"].nunique(),
    "orders_with_item_data": order_item_features["order_id"].nunique(),
    "delivered_orders_without_items": (
        ~delivered_orders["order_id"]
        .isin(order_item_features["order_id"])
    ).sum()
})

delivered_orders                  96478
orders_with_item_data             96478
delivered_orders_without_items        0
dtype: int64

In [67]:
payments = datasets["payments"].copy()

delivered_payments = payments[
    payments["order_id"].isin(delivered_orders["order_id"])
].copy()

print(f"All payment records:       {len(payments):,}")
print(f"Delivered payment records: {len(delivered_payments):,}")
print(
    f"Orders represented:        "
    f"{delivered_payments['order_id'].nunique():,}"
)

All payment records:       103,886
Delivered payment records: 100,756
Orders represented:        96,477


In [68]:
# Validate payment values

delivered_payments[
    ["payment_value", "payment_installments"]
].describe()

,payment_value,payment_installments
count,100756.000000,100756.000000
mean,153.067428,2.851632
std,214.451418,2.684378
min,0.000000,0.000000
25%,56.780000,1.000000
50%,100.000000,1.000000
75%,171.290000,4.000000
max,13664.080000,24.000000


In [69]:
payment_quality = pd.Series({
    "negative_payment_value":
        (delivered_payments["payment_value"] < 0).sum(),

    "zero_payment_value":
        (delivered_payments["payment_value"] == 0).sum(),

    "missing_payment_value":
        delivered_payments["payment_value"].isna().sum(),

    "negative_installments":
        (delivered_payments["payment_installments"] < 0).sum(),

    "zero_installments":
        (delivered_payments["payment_installments"] == 0).sum(),

    "missing_installments":
        delivered_payments["payment_installments"].isna().sum()
})

payment_quality

negative_payment_value    0
zero_payment_value        4
missing_payment_value     0
negative_installments     0
zero_installments         2
missing_installments      0
dtype: int64

In [70]:
# Check multiple payments per order

payments_per_order = (
    delivered_payments
    .groupby("order_id")
    .size()
)

payments_per_order.describe()

count    96477.000000
mean         1.044353
std          0.369603
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         26.000000
dtype: float64

In [71]:
print(
    "Orders with multiple payment records:",
    (payments_per_order > 1).sum()
)

print(
    "Maximum payment records for one order:",
    payments_per_order.max()
)

Orders with multiple payment records: 2875
Maximum payment records for one order: 26


In [ ]:
payment_quality

In [72]:
payments_per_order = (
    delivered_payments
    .groupby("order_id")
    .size()
)

payments_per_order.describe()

count    96477.000000
mean         1.044353
std          0.369603
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         26.000000
dtype: float64

In [73]:
print(
    "Orders with multiple payment records:",
    (payments_per_order > 1).sum()
)

print(
    "Maximum payment records for one order:",
    payments_per_order.max()
)

Orders with multiple payment records: 2875
Maximum payment records for one order: 26


In [74]:
print(
    "Orders with multiple payment records:",
    (payments_per_order > 1).sum()
)

print(
    "Maximum payment records for one order:",
    payments_per_order.max()
)

Orders with multiple payment records: 2875
Maximum payment records for one order: 26


In [76]:
delivered_without_payment = delivered_orders[
    ~delivered_orders["order_id"].isin(
        delivered_payments["order_id"]
    )
]

delivered_without_payment[
    ["order_id", "customer_id", "order_purchase_timestamp", "order_status"]
]

,order_id,customer_id,order_purchase_timestamp,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,2016-09-15 12:16:38,delivered


In [77]:
print(
    "Orders with multiple payment records:",
    (payments_per_order > 1).sum()
)

print(
    "Maximum payment records for one order:",
    payments_per_order.max()
)

Orders with multiple payment records: 2875
Maximum payment records for one order: 26


In [78]:
print(delivered_payments.columns.tolist())

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


In [79]:
payment_record_count=("payment_sequence", "count"),

In [80]:
order_payment_features = (
    delivered_payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "count"),
        payment_method_count=("payment_type", "nunique"),
        max_installments=("payment_installments", "max"),
        avg_installments=("payment_installments", "mean")
    )
    .reset_index()
)

order_payment_features.head()

,order_id,payment_value,payment_record_count,payment_method_count,max_installments,avg_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,3.0


In [81]:
pd.Series({
    "rows": len(order_payment_features),
    "unique_orders": order_payment_features["order_id"].nunique(),
    "duplicate_order_ids": order_payment_features["order_id"].duplicated().sum()
})

rows                   96477
unique_orders          96477
duplicate_order_ids        0
dtype: int64

In [82]:
# Validate payment aggregation

payment_validation = pd.Series({
    "raw_payment_total":
        delivered_payments["payment_value"].sum(),

    "aggregated_payment_total":
        order_payment_features["payment_value"].sum()
})

payment_validation

raw_payment_total           15422461.77
aggregated_payment_total    15422461.77
dtype: float64

In [83]:
pd.Series({
    "payment_rows": len(order_payment_features),
    "unique_order_ids": order_payment_features["order_id"].nunique(),
    "duplicate_order_ids": order_payment_features["order_id"].duplicated().sum()
})

payment_rows           96477
unique_order_ids       96477
duplicate_order_ids        0
dtype: int64

In [84]:
order_value_check = (
    order_item_features
    .merge(
        order_payment_features[
            ["order_id", "payment_value"]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

order_value_check["value_difference"] = (
    order_value_check["payment_value"]
    - order_value_check["order_value"]
)

In [85]:
order_value_check[
    ["order_value", "payment_value", "value_difference"]
].describe()

,order_value,payment_value,value_difference
count,96478.000000,96477.000000,96477.000000
mean,159.826839,159.856357,0.029349
std,218.794219,218.813144,1.138706
min,9.590000,9.590000,-51.620000
25%,61.850000,61.880000,0.000000
50%,105.280000,105.280000,0.000000
75%,176.260000,176.330000,0.000000
max,13664.080000,13664.080000,182.810000


In [86]:
value_reconciliation = pd.Series({
    "orders_compared":
        order_value_check["payment_value"].notna().sum(),

    "orders_without_payment":
        order_value_check["payment_value"].isna().sum(),

    "exact_match_within_1_cent":
        (
            order_value_check["value_difference"].abs() < 0.01
        ).sum(),

    "difference_over_1":
        (
            order_value_check["value_difference"].abs() > 1
        ).sum(),

    "max_absolute_difference":
        order_value_check["value_difference"].abs().max()
})

value_reconciliation

orders_compared              96477.00
orders_without_payment           1.00
exact_match_within_1_cent    96102.00
difference_over_1              246.00
max_absolute_difference        182.81
dtype: float64

In [87]:
reviews = datasets["reviews"].copy()

delivered_reviews = reviews[
    reviews["order_id"].isin(delivered_orders["order_id"])
].copy()

print(f"All review records:       {len(reviews):,}")
print(f"Delivered review records: {len(delivered_reviews):,}")
print(
    f"Orders represented:       "
    f"{delivered_reviews['order_id'].nunique():,}"
)

All review records:       99,224
Delivered review records: 96,361
Orders represented:       95,832


In [88]:
delivered_reviews["review_score"].describe()

count    96361.000000
mean         4.155717
std          1.284986
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

In [89]:
review_quality = pd.Series({
    "missing_review_score":
        delivered_reviews["review_score"].isna().sum(),

    "below_1":
        (delivered_reviews["review_score"] < 1).sum(),

    "above_5":
        (delivered_reviews["review_score"] > 5).sum(),

    "unique_review_ids":
        delivered_reviews["review_id"].nunique(),

    "review_records":
        len(delivered_reviews)
})

review_quality

missing_review_score        0
below_1                     0
above_5                     0
unique_review_ids       95647
review_records          96361
dtype: int64

In [90]:
reviews_per_order = (
    delivered_reviews
    .groupby("order_id")
    .size()
)

print(
    "Orders with multiple review records:",
    (reviews_per_order > 1).sum()
)

print(
    "Maximum review records for one order:",
    reviews_per_order.max()
)

print(
    "Delivered orders without reviews:",
    (
        ~delivered_orders["order_id"]
        .isin(delivered_reviews["order_id"])
    ).sum()
)

Orders with multiple review records: 525
Maximum review records for one order: 3
Delivered orders without reviews: 646


In [91]:
# Aggregate reviews to order level

# For multiple review records, we'll use the mean review score and preserve the number of review records.


order_review_features = (
    delivered_reviews
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_record_count=("review_id", "count")
    )
    .reset_index()
)

order_review_features.head()

,order_id,avg_review_score,review_record_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [92]:
review_aggregation_validation = pd.Series({
    "rows": len(order_review_features),
    "unique_orders": order_review_features["order_id"].nunique(),
    "duplicate_order_ids":
        order_review_features["order_id"].duplicated().sum(),

    "min_review_score":
        order_review_features["avg_review_score"].min(),

    "max_review_score":
        order_review_features["avg_review_score"].max()
})

review_aggregation_validation

rows                   95832.0
unique_orders          95832.0
duplicate_order_ids        0.0
min_review_score           1.0
max_review_score           5.0
dtype: float64

In [93]:
# Validate coverage


review_coverage = pd.Series({
    "delivered_orders":
        delivered_orders["order_id"].nunique(),

    "orders_with_reviews":
        order_review_features["order_id"].nunique(),

    "orders_without_reviews":
        (
            ~delivered_orders["order_id"]
            .isin(order_review_features["order_id"])
        ).sum(),

    "review_coverage_pct":
        (
            order_review_features["order_id"].nunique()
            / delivered_orders["order_id"].nunique()
            * 100
        )
})

review_coverage

delivered_orders          96478.000000
orders_with_reviews       95832.000000
orders_without_reviews      646.000000
review_coverage_pct          99.330417
dtype: float64

In [94]:
products = datasets["products"].copy()

print("Product rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())

products.head()

Product rows: 32951
Unique product IDs: 32951


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [95]:
product_category_quality = pd.Series({
    "total_products": len(products),

    "unique_products":
        products["product_id"].nunique(),

    "missing_category":
        products["product_category_name"].isna().sum(),

    "unique_categories":
        products["product_category_name"].nunique()
})

product_category_quality

total_products       32951
unique_products      32951
missing_category       610
unique_categories       73
dtype: int64

In [96]:
item_product_check = (
    delivered_order_items[
        ["order_id", "product_id"]
    ]
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

pd.Series({
    "delivered_item_records":
        len(item_product_check),

    "matched_products":
        (item_product_check["_merge"] == "both").sum(),

    "unmatched_product_ids":
        (item_product_check["_merge"] == "left_only").sum(),

    "items_with_missing_category":
        item_product_check["product_category_name"].isna().sum()
})

delivered_item_records         110197
matched_products               110197
unmatched_product_ids               0
items_with_missing_category      1537
dtype: int64

In [97]:
# Inspect category distribution


category_distribution = (
    item_product_check["product_category_name"]
    .value_counts(dropna=False)
    .head(15)
)

category_distribution

product_category_name
cama_mesa_banho           10953
beleza_saude               9465
esporte_lazer              8431
moveis_decoracao           8160
informatica_acessorios     7644
utilidades_domesticas      6795
relogios_presentes         5859
telefonia                  4430
ferramentas_jardim         4268
automotivo                 4140
brinquedos                 4030
cool_stuff                 3718
perfumaria                 3340
bebes                      2982
eletronicos                2729
Name: count, dtype: int64

In [98]:
# Category Distribution

category_distribution = (
    item_product_check["product_category_name"]
    .value_counts(dropna=False)
    .head(15)
)

category_distribution

product_category_name
cama_mesa_banho           10953
beleza_saude               9465
esporte_lazer              8431
moveis_decoracao           8160
informatica_acessorios     7644
utilidades_domesticas      6795
relogios_presentes         5859
telefonia                  4430
ferramentas_jardim         4268
automotivo                 4140
brinquedos                 4030
cool_stuff                 3718
perfumaria                 3340
bebes                      2982
eletronicos                2729
Name: count, dtype: int64

In [99]:
# Add categories to delivered items

delivered_items_enriched = (
    delivered_order_items
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)

print("Rows before merge:", len(delivered_order_items))
print("Rows after merge: ", len(delivered_items_enriched))

print(
    "Duplicate row inflation:",
    len(delivered_items_enriched)
    - len(delivered_order_items)
)

Rows before merge: 110197
Rows after merge:  110197
Duplicate row inflation: 0


In [100]:
# Build order-level category features


order_category_features = (
    delivered_items_enriched
    .groupby("order_id")
    .agg(
        category_count=(
            "product_category_name",
            "nunique"
        ),
        products_with_category=(
            "product_category_name",
            "count"
        )
    )
    .reset_index()
)

order_category_features.head()

,order_id,category_count,products_with_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1


In [101]:
category_coverage = pd.Series({
    "delivered_orders":
        delivered_orders["order_id"].nunique(),

    "orders_with_category_data":
        (order_category_features["category_count"] > 0).sum(),

    "orders_without_category_data":
        (order_category_features["category_count"] == 0).sum(),

    "max_categories_per_order":
        order_category_features["category_count"].max(),

    "avg_categories_per_order":
        order_category_features["category_count"].mean()
})

category_coverage

delivered_orders                96478.000000
orders_with_category_data       95146.000000
orders_without_category_data     1332.000000
max_categories_per_order            3.000000
avg_categories_per_order            0.993843
dtype: float64

In [102]:
order_base = (
    delivered_orders
    .merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

print("Rows:", len(order_base))
print("Unique orders:", order_base["order_id"].nunique())
print(
    "Missing customer_unique_id:",
    order_base["customer_unique_id"].isna().sum()
)

Rows: 96478
Unique orders: 96478
Missing customer_unique_id: 0


In [103]:
order_analytics = (
    order_base
    .merge(
        order_item_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [104]:
print("Rows after item join:", len(order_analytics))
print(
    "Duplicate order IDs:",
    order_analytics["order_id"].duplicated().sum()
)

Rows after item join: 96478
Duplicate order IDs: 0


In [105]:
# Join payment features

order_analytics = (
    order_analytics
    .merge(
        order_payment_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [106]:
# Join review features


order_analytics = (
    order_analytics
    .merge(
        order_review_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [107]:
# Join category features

order_analytics = (
    order_analytics
    .merge(
        order_category_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [108]:
# Critical integrity validation

order_analytics_validation = pd.Series({
    "rows":
        len(order_analytics),

    "unique_orders":
        order_analytics["order_id"].nunique(),

    "duplicate_order_ids":
        order_analytics["order_id"].duplicated().sum(),

    "unique_customers":
        order_analytics["customer_unique_id"].nunique(),

    "missing_customer":
        order_analytics["customer_unique_id"].isna().sum(),

    "missing_item_features":
        order_analytics["item_count"].isna().sum(),

    "missing_payment_features":
        order_analytics["payment_value"].isna().sum(),

    "missing_review_features":
        order_analytics["avg_review_score"].isna().sum(),

    "missing_category_features":
        order_analytics["category_count"].isna().sum()
})

order_analytics_validation

rows                         96478
unique_orders                96478
duplicate_order_ids              0
unique_customers             93358
missing_customer                 0
missing_item_features            0
missing_payment_features         1
missing_review_features        646
missing_category_features        0
dtype: int64

In [109]:
customer_features = (
    order_analytics
    .groupby("customer_unique_id")
    .agg(
        last_purchase_date=(
            "order_purchase_timestamp",
            "max"
        ),

        first_purchase_date=(
            "order_purchase_timestamp",
            "min"
        ),

        frequency=(
            "order_id",
            "nunique"
        ),

        monetary_value=(
            "order_value",
            "sum"
        ),

        avg_order_value=(
            "order_value",
            "mean"
        ),

        total_items=(
            "item_count",
            "sum"
        ),

        avg_items_per_order=(
            "item_count",
            "mean"
        )
    )
    .reset_index()
)

customer_features.head()

,customer_unique_id,last_purchase_date,first_purchase_date,frequency,monetary_value,avg_order_value,total_items,avg_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,141.90,141.90,1,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,27.19,27.19,1,1.0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,86.22,86.22,1,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,43.62,43.62,1,1.0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,196.89,196.89,1,1.0


In [110]:
customer_features["recency_days"] = (
    reference_date
    - customer_features["last_purchase_date"]
).dt.days

customer_features["tenure_days"] = (
    reference_date
    - customer_features["first_purchase_date"]
).dt.days

In [111]:
customer_features["recency_days"] = (
    reference_date
    - customer_features["last_purchase_date"]
).dt.days

customer_features["tenure_days"] = (
    reference_date
    - customer_features["first_purchase_date"]
).dt.days

In [112]:
# Validate customer grain


customer_feature_validation = pd.Series({
    "rows":
        len(customer_features),

    "unique_customers":
        customer_features["customer_unique_id"].nunique(),

    "duplicate_customer_ids":
        customer_features["customer_unique_id"].duplicated().sum(),

    "missing_customer_ids":
        customer_features["customer_unique_id"].isna().sum(),

    "min_frequency":
        customer_features["frequency"].min(),

    "max_frequency":
        customer_features["frequency"].max(),

    "negative_recency":
        (customer_features["recency_days"] < 0).sum(),

    "negative_monetary":
        (customer_features["monetary_value"] < 0).sum()
})

customer_feature_validation

rows                      93358
unique_customers          93358
duplicate_customer_ids        0
missing_customer_ids          0
min_frequency                 1
max_frequency                15
negative_recency              0
negative_monetary             0
dtype: int64

In [113]:
# Inspect core feature distributions

customer_features[
    [
        "recency_days",
        "frequency",
        "monetary_value",
        "avg_order_value",
        "total_items",
        "avg_items_per_order",
        "tenure_days"
    ]
].describe().T



,count,mean,std,min,25%,50%,75%,max
recency_days,93358.0,237.478888,152.595050,0.00,114.00,218.00,346.00,713.00
frequency,93358.0,1.033420,0.209097,1.00,1.00,1.00,1.00,15.00
monetary_value,93358.0,165.168210,226.292101,9.59,63.01,107.78,182.51,13664.08
avg_order_value,93358.0,160.287465,219.554565,9.59,62.33,105.63,176.59,13664.08
total_items,93358.0,1.180370,0.620857,1.00,1.00,1.00,1.00,24.00
avg_items_per_order,93358.0,1.139531,0.527075,1.00,1.00,1.00,1.00,21.00
tenure_days,93358.0,240.123974,153.102801,0.00,116.00,221.00,350.00,713.00


In [114]:
phase2_final_validation = pd.Series({
    "rows": len(order_analytics),
    "unique_orders": order_analytics["order_id"].nunique(),
    "duplicate_order_ids": order_analytics["order_id"].duplicated().sum(),
    "unique_customers": order_analytics["customer_unique_id"].nunique(),
    "missing_customer_ids": order_analytics["customer_unique_id"].isna().sum()
})

phase2_final_validation

rows                    96478
unique_orders           96478
duplicate_order_ids         0
unique_customers        93358
missing_customer_ids        0
dtype: int64

In [115]:
# Define the processed-data path
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(PROCESSED_DIR)

C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\data\processed


In [116]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [117]:
ORDER_ANALYTICS_PATH = PROCESSED_DIR / "order_analytics.parquet"

order_analytics.to_parquet(
    ORDER_ANALYTICS_PATH,
    index=False
)

print("Saved:", ORDER_ANALYTICS_PATH)

Saved: C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\data\processed\order_analytics.parquet


In [118]:
order_analytics_test = pd.read_parquet(
    ORDER_ANALYTICS_PATH
)

persisted_validation = pd.Series({
    "rows": len(order_analytics_test),

    "unique_orders":
        order_analytics_test["order_id"].nunique(),

    "duplicate_order_ids":
        order_analytics_test["order_id"].duplicated().sum(),

    "unique_customers":
        order_analytics_test["customer_unique_id"].nunique(),

    "missing_customer_ids":
        order_analytics_test["customer_unique_id"].isna().sum()
})

persisted_validation

rows                    96478
unique_orders           96478
duplicate_order_ids         0
unique_customers        93358
missing_customer_ids        0
dtype: int64

In [119]:
# Inspect the schema

order_analytics.info()

<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  str           
 1   customer_id                    96478 non-null  str           
 2   order_status                   96478 non-null  str           
 3   order_purchase_timestamp       96478 non-null  datetime64[us]
 4   order_approved_at              96464 non-null  datetime64[us]
 5   order_delivered_carrier_date   96476 non-null  datetime64[us]
 6   order_delivered_customer_date  96470 non-null  datetime64[us]
 7   order_estimated_delivery_date  96478 non-null  datetime64[us]
 8   purchase_year                  96478 non-null  int32         
 9   purchase_month                 96478 non-null  period[M]     
 10  customer_unique_id             96478 non-null  str           
 11  item_count                

In [120]:
print(order_analytics.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'customer_unique_id', 'item_count', 'product_count', 'seller_count', 'merchandise_value', 'freight_value', 'avg_item_price', 'order_value', 'payment_value', 'payment_record_count', 'payment_method_count', 'max_installments', 'avg_installments', 'avg_review_score', 'review_record_count', 'category_count', 'products_with_category']


In [122]:
# Validate the loaded Phase 2 artifact

load_validation = pd.Series({
    "rows":
        len(order_analytics),

    "unique_orders":
        order_analytics["order_id"].nunique(),

    "unique_customers":
        order_analytics["customer_unique_id"].nunique(),

    "duplicate_order_ids":
        order_analytics["order_id"].duplicated().sum(),

    "missing_customer_ids":
        order_analytics["customer_unique_id"].isna().sum()
})

load_validation

rows                    96478
unique_orders           96478
unique_customers        93358
duplicate_order_ids         0
missing_customer_ids        0
dtype: int64

In [123]:
# Verify timestamps

# We specifically need order_purchase_timestamp for Recency and Tenure.

pd.Series({
    "purchase_timestamp_dtype":
        str(order_analytics["order_purchase_timestamp"].dtype),

    "first_purchase":
        order_analytics["order_purchase_timestamp"].min(),

    "last_purchase":
        order_analytics["order_purchase_timestamp"].max(),

    "missing_purchase_timestamp":
        order_analytics["order_purchase_timestamp"].isna().sum()
})

purchase_timestamp_dtype           datetime64[us]
first_purchase                2016-09-15 12:16:38
last_purchase                 2018-08-29 15:00:37
missing_purchase_timestamp                      0
dtype: object